## Extração de UASGs de Pernambuco

Obtém a lista de todas as Unidades Administrativas de Serviços Gerais (UASGs) ativas
no estado de Pernambuco, via API de Dados Abertos do Compras.gov.br.

As UASGs são as unidades organizacionais que realizam compras públicas — prefeituras,
secretarias, autarquias, etc. A lista serve como ponto de entrada para identificar
quais órgãos têm licitações registradas no sistema federal.

Fluxo:
1. Consulta paginada à API de UASGs filtrando por `siglaUf=PE` e `statusUasg=true`
2. Acumula todos os resultados e salva em `data/raw/uasg_pe.csv`

Fonte:
- dadosabertos.compras.gov.br: `modulo-uasg/1_consultarUasg`

Saída:
- `data/raw/uasg_pe.csv` — lista de UASGs ativas em PE (código, nome, município)

In [1]:
import os; os.makedirs("data/raw", exist_ok=True)

### 1. Extração de UASGs Ativas

In [2]:
import urllib.request
import urllib.parse
import json
import pandas as pd
import time

def get_uasg_pe():
    base_url = "https://dadosabertos.compras.gov.br/modulo-uasg/1_consultarUasg"
    params = {"siglaUf": "PE", "statusUasg": "true", "pagina": 1}
    results = []
    while True:
        url = base_url + "?" + urllib.parse.urlencode(params)
        with urllib.request.urlopen(url, timeout=15) as resp:
            r = json.loads(resp.read().decode())
        page = r.get("resultado", [])
        results.extend(page)
        if len(page) < 500:  
            break
        params["pagina"] += 1
        time.sleep(0.2)
    return pd.DataFrame(results)

uasg_pe = get_uasg_pe()
uasg_pe.to_csv("data/raw/uasg_pe.csv", index=False)
print(f"{len(uasg_pe)} UASGs encontradas em PE")

1222 UASGs encontradas em PE
